<a href="https://colab.research.google.com/github/Anandxz/Machine-Learning/blob/main/cloumn_transfer_day_28.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import pandas as pd
import numpy as np

In [5]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

In [6]:
df=pd.read_csv("covid_toy.csv")

In [7]:
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [8]:
df.sample(3)

,age,gender,fever,cough,city,has_covid
11,65,Female,98.0,Mild,Mumbai,Yes
32,34,Female,101.0,Strong,Delhi,Yes
94,79,Male,NaN,Strong,Kolkata,Yes


In [9]:
df.shape

(100, 6)

In [10]:
df['age'].nunique()

55

In [11]:
df['cough'].value_counts()

,count
cough,
Mild,62
Strong,38


In [12]:
df.isnull().sum()

,0
age,0
gender,0
fever,10
cough,0
city,0
has_covid,0


In [13]:
from sklearn.model_selection import train_test_split


In [14]:
X_train,X_test,y_train,y_test = train_test_split(df.drop(columns=['has_covid']),df['has_covid'],test_size=0.2)

In [15]:
X_train

,age,gender,fever,cough,city
37,55,Male,100.0,Mild,Kolkata
25,23,Male,NaN,Mild,Mumbai
58,23,Male,98.0,Strong,Mumbai
81,65,Male,99.0,Mild,Delhi
24,13,Female,100.0,Strong,Kolkata
...,...,...,...,...,...
74,34,Female,104.0,Strong,Delhi
84,69,Female,98.0,Strong,Mumbai
48,66,Male,99.0,Strong,Bangalore
9,64,Female,101.0,Mild,Delhi


#Without Column Transformer

In [16]:
si=SimpleImputer()

In [17]:
X_train_fever=si.fit_transform(X_train[['fever']])
X_test_fever=si.fit_transform(X_test[['fever']])

X_train_fever.shape

#we are applying a simple in=mputer into the missing data with the mean value .

(80, 1)

In [18]:
#Ordinal Encoding over -->> Cough( because it has ranking optin)
oe=OrdinalEncoder(categories=[['Mild','Strong']])
X_train_cough=oe.fit_transform(X_train[['cough']])
X_test_cough=oe.fit_transform(X_test[['cough']])
X_train_cough.shape


(80, 1)

In [19]:
df['gender'].value_counts()

,count
gender,
Female,59
Male,41


In [20]:
#applying ONehotEncoding in gender
ohe=OneHotEncoder(drop='first') #drop first measn we remoce the first coloumns
#for the multi colinearity
X_train_gender_city=ohe.fit_transform(X_train[['gender','city']])
X_test_gender_city=ohe.fit_transform(X_test[['gender','city']])

X_train_gender_city.shape


(80, 4)

In [21]:
#Extracting the AGE
X_train_age=X_train.drop(columns=['gender','fever','cough','city'])
X_test_age=X_test.drop(columns=['gender','fever','cough','city'])

X_train_age.shape


(80, 1)

In [22]:
#NOw we have all the colomns \


In [23]:
X_train_transformed=np.concatenate((X_train_age,X_train_fever,X_train_gender_city.toarray(),X_train_cough),axis=1)


X_test_transformed=np.concatenate((X_test_age,X_test_fever,X_test_gender_city.toarray(),X_test_cough),axis=1)


X_train_transformed.shape

(80, 7)

In [25]:
df.sample(3)

,age,gender,fever,cough,city,has_covid
16,69,Female,103.0,Mild,Kolkata,Yes
30,15,Male,101.0,Mild,Delhi,Yes
73,34,Male,98.0,Strong,Kolkata,Yes


In [24]:
X_train_transformed


array([[ 55.  , 100.  ,   1.  ,   0.  ,   1.  ,   0.  ,   0.  ],
       [ 23.  , 100.75,   1.  ,   0.  ,   0.  ,   1.  ,   0.  ],
       [ 23.  ,  98.  ,   1.  ,   0.  ,   0.  ,   1.  ,   1.  ],
       [ 65.  ,  99.  ,   1.  ,   1.  ,   0.  ,   0.  ,   0.  ],
       [ 13.  , 100.  ,   0.  ,   0.  ,   1.  ,   0.  ,   1.  ],
       [  8.  , 101.  ,   0.  ,   0.  ,   1.  ,   0.  ,   0.  ],
       [ 59.  ,  99.  ,   0.  ,   1.  ,   0.  ,   0.  ,   1.  ],
       [ 51.  , 104.  ,   1.  ,   0.  ,   1.  ,   0.  ,   0.  ],
       [ 83.  , 101.  ,   0.  ,   0.  ,   1.  ,   0.  ,   0.  ],
       [ 75.  , 100.75,   0.  ,   1.  ,   0.  ,   0.  ,   0.  ],
       [ 15.  , 101.  ,   1.  ,   1.  ,   0.  ,   0.  ,   0.  ],
       [ 34.  , 101.  ,   0.  ,   1.  ,   0.  ,   0.  ,   1.  ],
       [ 10.  ,  98.  ,   0.  ,   0.  ,   1.  ,   0.  ,   1.  ],
       [ 33.  , 102.  ,   0.  ,   1.  ,   0.  ,   0.  ,   1.  ],
       [ 65.  ,  98.  ,   0.  ,   0.  ,   0.  ,   1.  ,   0.  ],
       [ 83.  ,  98.  ,  

#Mentos or Column tech

In [26]:
from sklearn.compose import ColumnTransformer

In [28]:
transformer= ColumnTransformer(transformers=[
    ('tnf1',SimpleImputer(),['fever']),
    ('tnf2',OrdinalEncoder(categories=[['Mild','Strong']]),['cough']),
    ('tnf3',OneHotEncoder(drop='first'),['gender','city'])
],remainder='passthrough')

In [31]:
transformer.fit_transform(X_train).shape

(80, 7)